<a href="https://colab.research.google.com/github/ashleyogorman/ICP_Milestone_1/blob/main/milestone_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ID Card OCR ICP Pipeline**


---



classifies which country an ID card belongs to

**Pipeline Stages**: image enhancement -> channel selection -> OCR



---


CS 8001 End to End Machine Learning

Ashley O'Gorman and Alexander Hardaway

# Setup
---

In [ ]:
import cv2
import numpy as np
import pytesseract
import easyocr
from PIL import Image
from datasets import load_dataset
import time

In [ ]:
print("--- Loading dataset 'lansinuote/ocr_id_card' ---")
try:
    ds = load_dataset("lansinuote/ocr_id_card", split='train')
    print("Dataset loaded successfully.")
    print(ds)
except Exception as e:
    print(f"Failed to load dataset: {e}")
    exit()

# Step 1: Image Enhancement Step


---


Contract: ***enhance_fn(image: np.ndarray) -> np.ndarray:***

The first step of the pipeline will be to use the cv2 library from python to enchance the base image if need be. The documentation to the library can be found here: https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html.

You may need to adjust the sharpness, brightness, or contrast of the image or make no changes at all. This is just a sample, as you are encouraged to find other helpful operations that can help increase the accuracy of the pipeline.

In [ ]:
# Baseline: no enhancement applied
def enhance_none(image: np.ndarray) -> np.ndarray:
    return image

Contrast

In [ ]:
# Ashley
def enhance_constrast_increase_25(image: np.ndarray) -> np.ndarray:
    contrast = 1.25
    brightness = 0
    return cv2.convertScaleAbs(image, alpha=contrast, beta=brightness)

In [ ]:
# Ashley
def enhance_constrast_increase_50(image: np.ndarray) -> np.ndarray:
    contrast = 1.5
    brightness = 0
    return cv2.convertScaleAbs(image, alpha=contrast, beta=brightness)

In [ ]:
# Ashley
def enhance_constrast_increase_75(image: np.ndarray) -> np.ndarray:
    contrast = 1.75
    brightness = 0
    return cv2.convertScaleAbs(image, alpha=contrast, beta=brightness)

Brightness

In [ ]:
# Ashley
def enhance_brightness_decrease_50(image: np.ndarray) -> np.ndarray:
    contrast = 1.0
    brightness = -50
    return cv2.convertScaleAbs(image, alpha=contrast, beta=brightness)

In [ ]:
# Ashley
def enhance_brightness_increase_50(image: np.ndarray) -> np.ndarray:
    contrast = 1.0
    brightness = 50
    return cv2.convertScaleAbs(image, alpha=contrast, beta=brightness)

In [ ]:
# Ashley
def enhance_brightness_increase_100(image: np.ndarray) -> np.ndarray:
    contrast = 1.0
    brightness = 100
    return cv2.convertScaleAbs(image, alpha=contrast, beta=brightness)

Laplacian

In [ ]:
# Ashley
def enhance_laplacian_simple(image: np.ndarray) -> np.ndarray:
    return cv2.Laplacian(image, -1)

In [ ]:
# Ashley
def enhance_laplacian_precise(image: np.ndarray) -> np.ndarray:
    laplacian_64f = cv2.Laplacian(image, cv2.CV_64F)
    laplacian_uint8 = np.uint8(np.absolute(laplacian_64f))
    return laplacian_uint8

Saturation

In [ ]:
# Alex
def enhance_saturation_increase_50(image: np.ndarray) -> np.ndarray:
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    h, s, v = cv2.split(hsv)
    s = cv2.convertScaleAbs(s, alpha=1.5, beta=0)
    return cv2.cvtColor(cv2.merge((h, s, v)), cv2.COLOR_HSV2RGB)

Sharpen

In [ ]:
# Alex
def enhance_sharpen(image: np.ndarray) -> np.ndarray:
    blurred = cv2.GaussianBlur(image, (0, 0), 3)
    return cv2.addWeighted(image, 1.5, blurred, -0.5, 0)

Deskew

In [ ]:
# Alex
def enhance_deskew(image: np.ndarray) -> np.ndarray:
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    card = cv2.inRange(hsv, (0, 0, 150), (180, 60, 255))
    contours, _ = cv2.findContours(card, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return image
    _, _, angle = cv2.minAreaRect(max(contours, key=cv2.contourArea))
    if angle > 45:
        angle -= 90
    height, width = image.shape[:2]
    rotation = cv2.getRotationMatrix2D((width / 2, height / 2), angle, 1.0)
    return cv2.warpAffine(image, rotation, (width, height), borderMode=cv2.BORDER_REPLICATE)

All enhancement options:

In [ ]:
enhancement_options = {
    "none": enhance_none,
    
    "+.25 contrast": enhance_constrast_increase_25,
    "+.50 contrast": enhance_constrast_increase_50,
    "+.75 contrast": enhance_constrast_increase_75,
    
    "-50 brightness": enhance_brightness_decrease_50,
    "+50 brightness": enhance_brightness_increase_50,
    "+100 brightness": enhance_brightness_increase_100,
    
    "simple laplacian": enhance_laplacian_simple,
    "precise laplacian": enhance_laplacian_precise,
    
    "+50% saturation": enhance_saturation_increase_50,
    "sharpen": enhance_sharpen,
    "deskew": enhance_deskew,
}

# Step 2: Channel Selection Step
---
Contract: ***channel_fn(image: np.ndarray) -> np.ndarray:***

The next step is choose to extract a specific color channel from the image array. You may choose one color or a combination of colors or the orignal image as well. As before you can use the cv2 library to do these operations.

In [ ]:
# Baseline: no chanel applied
def channel_none(image: np.ndarray) -> np.ndarray:
    return image

Single Colors

In [ ]:
# Ashley
def channel_blue(image: np.ndarray) -> np.ndarray:
    b, g, r = cv2.split(image)
    return b

In [ ]:
# Ashley
def channel_green(image: np.ndarray) -> np.ndarray:
    b, g, r = cv2.split(image)
    return g

In [ ]:
# Ashley
def channel_red(image: np.ndarray) -> np.ndarray:
    b, g, r = cv2.split(image)
    return r

Multicolors

In [ ]:
# Alex
def channel_red_green(image: np.ndarray) -> np.ndarray:
    return cv2.addWeighted(image[:, :, 0], 0.5, image[:, :, 1], 0.5, 0)

In [ ]:
# Alex
def channel_green_blue(image: np.ndarray) -> np.ndarray:
    return cv2.addWeighted(image[:, :, 1], 0.5, image[:, :, 2], 0.5, 0)

In [ ]:
# Alex
def channel_red_blue(image: np.ndarray) -> np.ndarray:
    return cv2.addWeighted(image[:, :, 0], 0.5, image[:, :, 2], 0.5, 0)

Grayscale

In [ ]:
# Alex
def channel_grayscale(image: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

In [ ]:
channel_options = {
    "none": channel_none,

    "blue": channel_blue,
    "green": channel_green,
    "red": channel_red,
    
    "red-green": channel_red_green,
    "green-blue": channel_green_blue,
    "red-blue": channel_red_blue,

    "grayscale": channel_grayscale,
}

# Step 3: OCR Step

---
Contract: ***ocr_fn(image: np.ndarray) -> str:***

The final step is to actually perform actual ocr. For this step, we pass the processed numpy array of the input image to the ocr models. To do this, we provide two initial ocr models:

EasyOcr

Tesseract

In [ ]:
# Ashley
def ocr_easyocr(image: np.ndarray) -> str:
    result = easy_ocr_reader.readtext(image)
    return result

In [ ]:
# Alex
import re

import numpy as np
import pytesseract

TESSERACT_LANG = "chi_sim"
TESSERACT_PSM = 11

_CJK_SPACE = re.compile(r"(?<=[\u4e00-\u9fff])\s+(?=[\u4e00-\u9fff])")


def ocr_tesseract(image: np.ndarray, psm: int = TESSERACT_PSM) -> list[dict]:
    data = pytesseract.image_to_data(
        image,
        lang=TESSERACT_LANG,
        config=f"--psm {psm}",
        output_type=pytesseract.Output.DICT,
    )
    lines = {}
    for i, word in enumerate(data["text"]):
        if not word.strip():
            continue
        key = (data["block_num"][i], data["par_num"][i], data["line_num"][i])
        x1, y1 = data["left"][i], data["top"][i]
        x2, y2 = x1 + data["width"][i], y1 + data["height"][i]
        if key not in lines:
            lines[key] = {"words": [], "box": [x1, y1, x2, y2], "confs": []}
        line = lines[key]
        line["words"].append(word)
        line["box"] = [
            min(line["box"][0], x1),
            min(line["box"][1], y1),
            max(line["box"][2], x2),
            max(line["box"][3], y2),
        ]
        line["confs"].append(float(data["conf"][i]))

    results = []
    for line in lines.values():
        results.append({
            "box": line["box"],
            "word": _CJK_SPACE.sub("", " ".join(line["words"])),
            "conf": round(sum(line["confs"]) / len(line["confs"]) / 100, 2),
        })
    return results

In [ ]:
ocr_options = {
    "tesseract": ocr_tesseract,
    "easyocr": ocr_easyocr,
}

# Assertion


---
Check the accuracy of the pipeline.


In [ ]:
# Alex
import datetime
import re
from collections import Counter

FIELD_NAMES = ["name", "sex", "ethnicity", "year", "month", "day", "address", "id_number"]


def ocr_lines(result) -> list[str]:
    lines = []
    for r in result:
        if isinstance(r, dict):
            lines.append(r["word"])
        elif isinstance(r, str):
            lines.append(r)
        else:
            lines.append(r[1])
    return lines


def find_id_number(lines: list[str]):
    for line in lines:
        match = re.search(r"(?<!\d)\d{17}[\dXx](?!\d)", "".join(line.split()))
        if match:
            return match.group()
    return None


def find_date(lines: list[str]):
    numbers = [int(n) for line in lines if find_id_number([line]) is None for n in re.findall(r"\d+", line)]
    for i in range(len(numbers) - 2):
        if 1900 <= numbers[i] <= 2030:
            try:
                return datetime.date(numbers[i], numbers[i + 1], numbers[i + 2])
            except (ValueError, OverflowError):
                return None
    return None


def run_assertions(result) -> dict:
    lines = ocr_lines(result)
    text = "".join("".join(lines).split())
    id_number = find_id_number(lines)
    date = find_date(lines)
    return {
        "date_valid": date is not None,
        "id_format": id_number is not None,
        "id_matches_date": id_number is not None and date is not None and id_number[6:14] == date.strftime("%Y%m%d"),
        "sex_valid": ("男" in text) != ("女" in text),
    }


def check_fields(result, target) -> dict:
    lines = ocr_lines(result)
    text = "".join("".join(lines).split())
    lines_without_id = [line for line in lines if find_id_number([line]) is None]
    numbers = Counter(int(n) for line in lines_without_id for n in re.findall(r"\d+", line))
    fields = {}
    for t in target:
        name = FIELD_NAMES[t["cls"]]
        value = "".join(t["word"].split())
        if name in ("year", "month", "day"):
            fields[name] = numbers[int(value)] > 0
            if fields[name]:
                numbers[int(value)] -= 1
        else:
            fields[name] = value in text
    return fields

# Pipeline


---
Method to run each pipeline.


In [ ]:
def run_pipeline(image, enhance_fn, channel_fn, ocr_fn):
    start = time.perf_counter()
    x = enhance_fn(image)
    x = channel_fn(x)
    img_text = ocr_fn(x)
    latency = time.perf_counter() - start
    return img_text, latency

# Run all configurations


---
Run all pipeline configurations.


In [ ]:
# This can take a while, only needed once
easy_ocr_reader = easyocr.Reader(['ch_sim'], gpu=False) #KVM@TACC is cpu only

In [ ]:
pipeline_results = []
for e_name, e_fn in enhancement_options.items():
    for c_name, c_fn in channel_options.items():
        for o_name, o_fn in ocr_options.items():
            for item in ds.select(range(10)): #placeholder for speed/ dev only
            # for item in ds: #uncomment for full pipeline run
                
                np_image = np.array(item['image'])
                true_image_result = item['ocr']
                pipeline_image_result, latency = run_pipeline(np_image, e_fn, c_fn, o_fn)
                    # current contract for pipeline_image_result format is a string
                    # if can have in similar format as true_image_results, that would be ideal
                    # easyocr can have detail=0 or 1 so can determine whats best based on tesseract abilities
                # print(pipeline_image_result)
                pipeline_results.append({
                    "target": true_image_result,
                    "result": pipeline_image_result,
                    "latency": latency,
                    "enhancement": e_name,
                    "channel": c_name,
                    "ocr": o_name,
                })
                # probably will want to add more above once we have accuracy metrics
                    
                print(latency)
                

# Display results as table


---
Display #TODO.


In [ ]:
number_of_pipelines = len(enhancement_options) * len(channel_options) * len(ocr_options)
print(f"Number of distinct pipeline configurations: {len(enhancement_options)} enhancement x {len(channel_options)} channel x {len(ocr_options)} OCR = {number_of_pipelines}")

In [ ]:
# Alex
import pandas as pd

rows = []
for r in pipeline_results:
    assertions = run_assertions(r["result"])
    fields = check_fields(r["result"], r["target"])
    rows.append({
        "enhancement": r["enhancement"],
        "channel": r["channel"],
        "ocr": r["ocr"],
        "latency": r["latency"],
        "assertions_passed": sum(assertions.values()),
        "assertions_total": len(assertions),
        "fields_correct": sum(fields.values()),
        "fields_total": len(fields),
        **assertions,
        **{f"{name}_correct": correct for name, correct in fields.items()},
    })
card_results = pd.DataFrame(rows)

grouped = card_results.groupby(["enhancement", "channel", "ocr"], sort=False)
results_table = pd.DataFrame({
    "cards": grouped.size(),
    "assertions passed": grouped["assertions_passed"].sum(),
    "assertions total": grouped["assertions_total"].sum(),
    "fields correct (%)": (100 * grouped["fields_correct"].sum() / grouped["fields_total"].sum()).round(1),
    "mean latency (s)": grouped["latency"].mean().round(3),
    "SLA met": "TBD",
}).reset_index()
results_table = results_table.sort_values(["assertions passed", "fields correct (%)"], ascending=False, ignore_index=True)

card_results.to_csv("card_results.csv", index=False)
results_table.to_csv("results_table.csv", index=False)

pd.set_option("display.max_rows", None)
results_table